In [11]:
import pandas as pd

In [12]:
mislabel = pd.read_csv(
    "data/cox2_th_final_missing_05_13_02/mislabel-detection.csv"
)

meta = pd.read_csv(
    "data/cox2_th_final_missing_05_13_02/output/cox2_th-meta.tsv",
    sep="\t",
    usecols=["accession", "genus"],
)

In [13]:
mislabeled = mislabel[mislabel["is_mislabeled"]].copy()

mislabeled = mislabeled.merge(
    meta, left_on="seq_id", right_on="accession", how="left"
)

result = mislabeled[["seq_id", "genus", "predicted_label", "score"]].rename(
    columns={
        "seq_id": "accession",
        "genus": "original_label",
        "predicted_label": "predicted_label",
        "score": "confidence_score",
    }
)

result["predicted_label"] = result["predicted_label"].str.removeprefix("g__")

result

,accession,original_label,predicted_label,confidence_score
0,AF086693.1,Leptolegnia,Aphanomyces,1.000000
1,AF290309.1,Lagenidium,Pythium,1.000000
2,AF290310.1,Myzocytiopsis,Lagenidium,1.000000
3,AF290311.1,Pythium,Halophytophthora,1.000000
4,HQ721468.1,Pythium,Phytopythium,1.000000
5,KJ595414.1,Pythium,Globisporangium,1.000000
6,KM819510.1,Pythiogeton,Globisporangium,1.000000
7,KY986681.1,Peronospora,Hyaloperonospora,1.000000
8,MG846712.1,Perofascia,Albugo,1.000000
9,NC_022178.1,Achlya,Saprolegnia,0.833333


In [14]:
sativa_df = pd.read_csv('/home/thomasH/code/sativa/cox2_th_v2/mislabeled-sequences.csv')


In [17]:
merged = result.merge(sativa_df, how='outer', suffixes=('_knn', '_sativa'), on='accession')

# For rows only in sativa (NaN in knn columns), fill from the input files
knn_lookup = mislabel[["seq_id", "predicted_label", "score"]].copy()
knn_lookup["predicted_label"] = knn_lookup["predicted_label"].str.removeprefix("g__")
knn_lookup = knn_lookup.rename(columns={"seq_id": "accession"})

meta_lookup = meta.rename(columns={"genus": "original_label_fill"})

merged = merged.merge(meta_lookup, on="accession", how="left")
merged["original_label_knn"] = merged["original_label_knn"].fillna(merged["original_label_fill"])
merged = merged.drop(columns=["original_label_fill"])

merged = merged.merge(knn_lookup, on="accession", how="left")
merged["predicted_label_knn"] = merged["predicted_label_knn"].fillna(merged["predicted_label"])
merged["confidence_score"] = merged["confidence_score"].fillna(merged["score"])
merged = merged.drop(columns=["predicted_label", "score"])

merged["mislabeled_knn"] = (
    merged["original_label_knn"].notna()
    & merged["predicted_label_knn"].notna()
    & (merged["original_label_knn"] != merged["predicted_label_knn"])
)
merged["mislabeled_sativa"] = (
    merged["original_label_sativa"].notna()
    & merged["predicted_label_sativa"].notna()
    & (merged["original_label_sativa"] != merged["predicted_label_sativa"])
)

merged

,accession,original_label_knn,predicted_label_knn,confidence_score,original_label_sativa,predicted_label_sativa,confidence,mislabeled_knn,mislabeled_sativa
0,AF086693.1,Leptolegnia,Aphanomyces,1.000000,Saprolegniaceae,Verrucalvaceae,1.000,True,True
1,AF290309.1,Lagenidium,Pythium,1.000000,Lagenidium,Pythium,1.000,True,True
2,AF290310.1,Myzocytiopsis,Lagenidium,1.000000,Myzocytiopsidales,Peronosporomycetes,0.999,True,True
3,AF290311.1,Pythium,Halophytophthora,1.000000,NaN,NaN,NaN,True,False
4,EU826093.1,Albugo,NaN,NaN,Albugo,Wilsoniana,0.968,False,True
5,GU360982.1,Plasmoverna,NaN,NaN,Plasmoverna,Plasmopara,0.663,False,True
6,HQ721468.1,Pythium,Phytopythium,1.000000,Pythium,Phytopythium,1.000,True,True
7,KJ595414.1,Pythium,Globisporangium,1.000000,Pythium,Globisporangium,0.999,True,True
8,KM819510.1,Pythiogeton,Globisporangium,1.000000,NaN,NaN,NaN,True,False
9,KT257465.1,Myzocytiopsis,NaN,NaN,Myzocytiopsidales,Peronosporomycetes,0.994,False,True
